In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

# Regression models
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

# Evaluation metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import mlflow
import mlflow.sklearn

import dagshub
import os

In [5]:
import dagshub

dagshub.init(
    repo_owner="yasmitha2211",
    repo_name="mlflow-project",
    mlflow=True
)

Initialized MLflow to track repo "yasmitha2211/mlflow-project"

Repository yasmitha2211/mlflow-project initialized!

In [6]:
mlflow.set_experiment(
    "Housing Price Prediction"
)

2026/08/06 13:28:10 INFO mlflow.tracking.fluent: Experiment with name 'Housing Price Prediction' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/4b6b314b807e4e5aab0c756d9b530c10', creation_time=1786003090817, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1786003090817, lifecycle_stage='active', name='Housing Price Prediction', tags={}, trace_location=None, workspace='default'>

In [9]:
# Load the housing dataset
import pandas as pd

data = pd.read_csv(r"C:\Users\Admin\Downloads\HousingData.csv")

# Fill missing values with the column mean
data = data.fillna(data.mean())

# Features (input variables)
X = data.drop("MEDV", axis=1)

# Target (house price)
y = data["MEDV"]

print("Feature Shape:", X.shape)
print("Target Shape:", y.shape)

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42
)

print("Training Features:", X_train.shape)
print("Testing Features:", X_test.shape)
print("Training Target:", y_train.shape)
print("Testing Target:", y_test.shape)

Feature Shape: (506, 13)
Target Shape: (506,)
Training Features: (354, 13)
Testing Features: (152, 13)
Training Target: (354,)
Testing Target: (152,)


In [10]:
# Build Regression Models

models = [
    (
        "Linear Regression",
        LinearRegression()
    ),
    (
        "Decision Tree",
        DecisionTreeRegressor(
            random_state=42
        )
    ),
    (
        "Random Forest",
        RandomForestRegressor(
            n_estimators=100,
            random_state=42
        )
    )
]

trained_models = []

for model_name, model in models:

    # Train the model
    model.fit(X_train, y_train)

    # Predict on test data
    predictions = model.predict(X_test)

    # Calculate evaluation metrics
    mae = mean_absolute_error(y_test, predictions)
    mse = mean_squared_error(y_test, predictions)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, predictions)

    trained_models.append(model)

    print("=" * 50)
    print(model_name)
    print("=" * 50)

    print("MAE :", mae)
    print("MSE :", mse)
    print("RMSE:", rmse)
    print("R2 Score:", r2)

Linear Regression
MAE : 3.141052567108417
MSE : 21.818457953525755
RMSE: 4.671023223398247
R2 Score: 0.7071862632031795
Decision Tree
MAE : 2.9388157894736846
MSE : 16.15677631578947
RMSE: 4.019549267740038
R2 Score: 0.7831686337460806
Random Forest
MAE : 2.137046052631579
MSE : 10.070587085526316
RMSE: 3.173418832351998
R2 Score: 0.8648480913485341


In [11]:
# Log All Experiments to DagsHub

for model_name, model in models:

    # Predict on test data
    predictions = model.predict(X_test)

    # Calculate metrics
    mae = mean_absolute_error(y_test, predictions)
    mse = mean_squared_error(y_test, predictions)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, predictions)

    with mlflow.start_run(run_name=model_name):

        # Log model name
        mlflow.log_param("Model", model_name)

        # Log model parameters
        mlflow.log_params(model.get_params())

        # Log evaluation metrics
        mlflow.log_metric("MAE", mae)
        mlflow.log_metric("MSE", mse)
        mlflow.log_metric("RMSE", rmse)
        mlflow.log_metric("R2_Score", r2)

        # Save the trained model
        mlflow.sklearn.log_model(
            model,
            "model"
        )

print("All Housing Regression Experiments Logged Successfully!")

2026/08/06 13:36:15 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Linear Regression at: https://dagshub.com/yasmitha2211/mlflow-project.mlflow/#/experiments/0/runs/dbad86afaf914ffd9a7a64846e05d796
🧪 View experiment at: https://dagshub.com/yasmitha2211/mlflow-project.mlflow/#/experiments/0


2026/08/06 13:36:47 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Decision Tree at: https://dagshub.com/yasmitha2211/mlflow-project.mlflow/#/experiments/0/runs/188f4666da2e4f7a8cc6bb283b052f72
🧪 View experiment at: https://dagshub.com/yasmitha2211/mlflow-project.mlflow/#/experiments/0


2026/08/06 13:37:11 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Random Forest at: https://dagshub.com/yasmitha2211/mlflow-project.mlflow/#/experiments/0/runs/865ddcc4de1f4ea0a1a3032fabb7c7d8
🧪 View experiment at: https://dagshub.com/yasmitha2211/mlflow-project.mlflow/#/experiments/0
All Housing Regression Experiments Logged Successfully!
